# Etapa 3 - Analise descritiva dos dados

Este notebook reproduz a segunda parte do trabalho: leitura da base consolidada, verificacao de estrutura, tratamento descritivo das variaveis do modelo, matriz de correlacao e visualizacoes com Plotly.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", "{:.6f}".format)

## 1. Caminhos e leitura da base

In [2]:
def localizar_material_entrega() -> Path:
    cwd = Path.cwd().resolve()
    candidatos = [
        cwd,
        cwd.parent,
        cwd / "Etapa3" / "Material_entrega",
        cwd.parent.parent,
    ]
    for candidato in candidatos:
        if (candidato / "outputs" / "base_etapa3_ms.csv").exists():
            return candidato
    raise FileNotFoundError("Nao encontrei outputs/base_etapa3_ms.csv a partir do diretorio atual.")


MATERIAL_ENTREGA = localizar_material_entrega()
OUTPUTS = MATERIAL_ENTREGA / "outputs"
BASE_PATH = OUTPUTS / "base_etapa3_ms.csv"

base = pd.read_csv(BASE_PATH, sep=";", decimal=",")
base.head()

,ano,cod_ibge,municipio,uf,populacao,pib,pib_per_capita,pib_per_capita_ibge,receita_corrente,despesa_pessoal,investimento,crescimento_pib,crescimento_populacao,log_pib_per_capita,receita_corrente_pib,gasto_pessoal_pib,investimento_pib,dummy_pequeno_municipio,interacao_pequeno_pessoal,area_km2,area_por_habitante,distancia_capital_km
0,2019,5000252,Alcinópolis,MS,5343,167819985.000000,31409.317799,31409.320000,46557057.740000,19509065.720000,7589401.040000,0.001584,0.014237,10.354860,0.277423,0.116250,0.045223,1,0.116250,4397.518000,0.823043,310.710000
1,2019,5000609,Amambai,MS,39396,972520383.000000,24685.764621,24685.760000,155044866.040000,81336214.280000,10289758.270000,-0.052290,0.011243,10.113982,0.159426,0.083634,0.010581,0,0.000000,4193.742000,0.106451,354.220000
2,2019,5000708,Anastácio,MS,25135,500752223.000000,19922.507380,19922.510000,71062909.920000,32680643.980000,7194346.470000,0.100068,0.000279,9.899605,0.141912,0.065263,0.014367,0,0.000000,2913.177000,0.115901,139.690000
3,2019,5000807,Anaurilândia,MS,9035,261086194.000000,28897.199115,28897.200000,45478282.400000,22390415.070000,13646128.620000,-0.001938,0.004670,10.271500,0.174189,0.085759,0.052267,0,0.000000,3415.657000,0.378047,368.540000
4,2019,5000856,Angélica,MS,10780,745122649.000000,69120.839425,69120.840000,53623518.440000,25835109.440000,3942004.680000,-0.097533,0.015066,11.143612,0.071966,0.034672,0.005290,0,0.000000,1283.627000,0.119075,272.510000


In [3]:
base.columns

Index(['ano', 'cod_ibge', 'municipio', 'uf', 'populacao', 'pib',
       'pib_per_capita', 'pib_per_capita_ibge', 'receita_corrente',
       'despesa_pessoal', 'investimento', 'crescimento_pib',
       'crescimento_populacao', 'log_pib_per_capita', 'receita_corrente_pib',
       'gasto_pessoal_pib', 'investimento_pib', 'dummy_pequeno_municipio',
       'interacao_pequeno_pessoal', 'area_km2', 'area_por_habitante',
       'distancia_capital_km'],
      dtype='str')

In [4]:
print(f"Linhas: {base.shape[0]}")
print(f"Colunas: {base.shape[1]}")
print(f"Municipios: {base['cod_ibge'].nunique()}")
print(f"Anos: {sorted(base['ano'].unique())}");

Linhas: 395
Colunas: 22
Municipios: 79
Anos: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]


## 2. Estrutura, tipos e valores ausentes

In [5]:
estrutura = pd.DataFrame({
    "coluna": base.columns,
    "tipo": [str(dtype) for dtype in base.dtypes],
    "ausentes": base.isna().sum().values,
    "ausentes_%": (base.isna().mean().values * 100).round(2),
    "valores_unicos": base.nunique(dropna=True).values,
})
estrutura

,coluna,tipo,ausentes,ausentes_%,valores_unicos
0,ano,int64,0,0.000000,5
1,cod_ibge,int64,0,0.000000,79
2,municipio,str,0,0.000000,79
3,uf,str,0,0.000000,1
4,populacao,int64,0,0.000000,313
5,pib,float64,0,0.000000,395
6,pib_per_capita,float64,0,0.000000,395
7,pib_per_capita_ibge,float64,0,0.000000,395
8,receita_corrente,float64,0,0.000000,394
9,despesa_pessoal,float64,0,0.000000,395


In [6]:
ausentes_df = (
    estrutura.query("ausentes > 0")
    .assign(pct=lambda d: (d["ausentes"] / len(base) * 100).round(1))
    .sort_values("ausentes", ascending=False)
)
fig = px.bar(
    ausentes_df,
    x="coluna",
    y="ausentes",
    text=ausentes_df["pct"].map("{:.1f}%".format),
    color="ausentes",
    color_continuous_scale=["#f7dc6f", "#e74c3c"],
    title=f"Valores ausentes por coluna (total de observações: {len(base)})",
    labels={"coluna": "Variável", "ausentes": "Ausentes (n)"},
)
fig.update_traces(textposition="outside")
fig.update_layout(
    coloraxis_showscale=False,
    xaxis_title="Variável",
    yaxis_title="Quantidade de ausentes",
    plot_bgcolor="white",
    yaxis=dict(gridcolor="#eeeeee"),
)
fig.show()

## 3. Variaveis analisadas

As estatisticas descritivas serao calculadas para as variaveis que compoem ou apoiam o modelo econometrico.

In [7]:
variaveis_modelo = [
    "crescimento_pib",
    "log_pib_per_capita",
    "investimento_pib",
    "gasto_pessoal_pib",
    "receita_corrente_pib",
    "crescimento_populacao",
    "area_por_habitante",
    "distancia_capital_km",
    "dummy_pequeno_municipio",
    "interacao_pequeno_pessoal",
]

variaveis_apoio = [
    "populacao",
    "pib",
    "pib_per_capita",
    "receita_corrente",
    "despesa_pessoal",
    "investimento",
    "area_km2",
]

variaveis_descritivas = variaveis_modelo + variaveis_apoio
base[variaveis_descritivas].describe().T

,count,mean,std,min,25%,50%,75%,max
crescimento_pib,395.000000,0.154473,0.195128,-0.581693,0.030835,0.141775,0.266688,1.094741
log_pib_per_capita,395.000000,10.786048,0.594605,9.377789,10.384542,10.728418,11.140804,12.925276
investimento_pib,395.000000,0.013873,0.011431,0.000901,0.006020,0.010871,0.017199,0.072569
gasto_pessoal_pib,395.000000,0.065878,0.029078,0.014374,0.042992,0.061487,0.083632,0.166429
receita_corrente_pib,395.000000,0.142189,0.065782,0.032564,0.092413,0.128942,0.173537,0.360413
crescimento_populacao,395.000000,0.007678,0.077285,-0.279889,-0.002422,0.008072,0.017795,0.388675
area_por_habitante,395.000000,0.262075,0.271332,0.008824,0.086312,0.161212,0.380056,1.599453
distancia_capital_km,395.000000,277.688861,117.258874,0.000000,198.000000,289.240000,368.540000,469.410000
dummy_pequeno_municipio,395.000000,0.253165,0.435376,0.000000,0.000000,0.000000,1.000000,1.000000
interacao_pequeno_pessoal,395.000000,0.017661,0.034275,0.000000,0.000000,0.000000,0.014533,0.136153


## 4. Estatisticas descritivas por ano

> **Nota sobre a moda:** Para variáveis contínuas com valores praticamente únicos, `pd.Series.mode()` retorna o primeiro valor da série ordenada (geralmente o mínimo). O valor é calculado por completude da tabela, mas não tem interpretação econômica relevante para variáveis contínuas. A moda é mais informativa para variáveis discretas, como `dummy_pequeno_municipio`.

In [8]:
def moda_serie(serie: pd.Series):
    moda = serie.dropna().mode()
    return np.nan if moda.empty else moda.iloc[0]


def erro_padrao(serie: pd.Series) -> float:
    serie = serie.dropna()
    if len(serie) == 0:
        return np.nan
    return serie.std(ddof=1) / np.sqrt(len(serie))


def estatisticas_grupo(df: pd.DataFrame, grupo=None) -> pd.DataFrame:
    linhas = []
    agrupado = [("base_empilhada", df)] if grupo is None else df.groupby(grupo)
    for chave, dados in agrupado:
        for variavel in variaveis_descritivas:
            serie = dados[variavel]
            linhas.append({
                "grupo": grupo or "total",
                "valor_grupo": chave,
                "variavel": variavel,
                "media": serie.mean(),
                "erro_padrao": erro_padrao(serie),
                "moda": moda_serie(serie),
                "mediana": serie.median(),
                "q1": serie.quantile(0.25),
                "q3": serie.quantile(0.75),
                "variancia": serie.var(ddof=1),
                "desvio_padrao": serie.std(ddof=1),
                "minimo": serie.min(),
                "maximo": serie.max(),
                "soma": serie.sum(),
                "contagem": serie.count(),
                "ausentes": serie.isna().sum(),
            })
    return pd.DataFrame(linhas)


estatisticas_total = estatisticas_grupo(base)
estatisticas_ano = estatisticas_grupo(base, "ano")
estatisticas_descritivas = pd.concat([estatisticas_total, estatisticas_ano], ignore_index=True)

estatisticas_path = OUTPUTS / "estatisticas_descritivas.csv"
estatisticas_descritivas.to_csv(estatisticas_path, index=False, sep=";", decimal=",", encoding="utf-8-sig")
estatisticas_descritivas.head(15)

,grupo,valor_grupo,variavel,media,erro_padrao,moda,mediana,q1,q3,variancia,desvio_padrao,minimo,maximo,soma,contagem,ausentes
0,total,base_empilhada,crescimento_pib,0.154473,0.009818,-0.581693,0.141775,0.030835,0.266688,0.038075,0.195128,-0.581693,1.094741,61.016676,395,0
1,total,base_empilhada,log_pib_per_capita,10.786048,0.029918,9.377789,10.728418,10.384542,11.140804,0.353555,0.594605,9.377789,12.925276,4260.488835,395,0
2,total,base_empilhada,investimento_pib,0.013873,0.000575,0.000901,0.010871,0.006020,0.017199,0.000131,0.011431,0.000901,0.072569,5.479853,395,0
3,total,base_empilhada,gasto_pessoal_pib,0.065878,0.001463,0.014374,0.061487,0.042992,0.083632,0.000846,0.029078,0.014374,0.166429,26.021789,395,0
4,total,base_empilhada,receita_corrente_pib,0.142189,0.003310,0.032564,0.128942,0.092413,0.173537,0.004327,0.065782,0.032564,0.360413,56.164744,395,0
5,total,base_empilhada,crescimento_populacao,0.007678,0.003889,0.000000,0.008072,-0.002422,0.017795,0.005973,0.077285,-0.279889,0.388675,3.032904,395,0
6,total,base_empilhada,area_por_habitante,0.262075,0.013652,0.293264,0.161212,0.086312,0.380056,0.073621,0.271332,0.008824,1.599453,103.519576,395,0
7,total,base_empilhada,distancia_capital_km,277.688861,5.899934,0.000000,289.240000,198.000000,368.540000,13749.643424,117.258874,0.000000,469.410000,109687.100000,395,0
8,total,base_empilhada,dummy_pequeno_municipio,0.253165,0.021906,0.000000,0.000000,0.000000,1.000000,0.189552,0.435376,0.000000,1.000000,100.000000,395,0
9,total,base_empilhada,interacao_pequeno_pessoal,0.017661,0.001725,0.000000,0.000000,0.000000,0.014533,0.001175,0.034275,0.000000,0.136153,6.975927,395,0


In [9]:
print(f"Estatisticas salvas em: {estatisticas_path}")

Estatisticas salvas em: C:\Users\marco\Documents\PUC-MINAS\4º Semestre\Eixo 4 - Projeto Mini Ministério da Fazenda\Projeto-Mini-Min-Fazenda\Etapa3\Material_entrega\outputs\estatisticas_descritivas.csv


## 5. Evolucao anual de indicadores selecionados

In [10]:
LABELS_SERIES = {
    "crescimento_pib_medio":      "Crescimento do PIB",
    "gasto_pessoal_pib_medio":    "Gasto c/ Pessoal / PIB",
    "investimento_pib_medio":     "Investimento / PIB",
    "receita_corrente_pib_media": "Receita Corrente / PIB",
}

indicadores_anuais = (
    base.groupby("ano", as_index=False)
    .agg(
        crescimento_pib_medio=("crescimento_pib", "mean"),
        gasto_pessoal_pib_medio=("gasto_pessoal_pib", "mean"),
        investimento_pib_medio=("investimento_pib", "mean"),
        receita_corrente_pib_media=("receita_corrente_pib", "mean"),
    )
)
indicadores_long = indicadores_anuais.melt(
    id_vars="ano", var_name="indicador", value_name="valor"
)
indicadores_long["indicador"] = indicadores_long["indicador"].map(LABELS_SERIES)

fig = px.line(
    indicadores_long,
    x="ano",
    y="valor",
    color="indicador",
    markers=True,
    title="Evolução anual — médias municipais (MS, 2019–2023)",
    labels={"indicador": "Indicador", "valor": "Média", "ano": "Ano"},
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.update_traces(marker_size=9, line_width=2.5)
fig.add_vline(
    x=2020, line_dash="dot", line_color="gray", line_width=1.5,
    annotation_text="2020 (COVID-19)",
    annotation_position="top left",
    annotation_font_size=11,
)
fig.update_yaxes(tickformat=".1%", gridcolor="#eeeeee")
fig.update_xaxes(tickmode="linear", dtick=1)
fig.update_layout(
    plot_bgcolor="white",
    legend=dict(orientation="h", yanchor="bottom", y=-0.3, xanchor="left", x=0),
    yaxis_title="Média (proporção do PIB / taxa)",
    xaxis_title="Ano",
)
fig.show()

In [11]:

# Evolucao das taxas de crescimento: PIB e populacao
dados_taxas = (
    base.groupby("ano", as_index=False)
    .agg(
        crescimento_pib=("crescimento_pib", "mean"),
        crescimento_populacao=("crescimento_populacao", "mean"),
    )
)
dados_taxas_long = dados_taxas.melt(id_vars="ano", var_name="indicador", value_name="valor")
dados_taxas_long["indicador"] = dados_taxas_long["indicador"].map({
    "crescimento_pib":       "Crescimento do PIB",
    "crescimento_populacao": "Crescimento Populacional",
})

fig_taxas = px.line(
    dados_taxas_long,
    x="ano", y="valor", color="indicador", markers=True,
    title="Evolucao anual — taxas de crescimento medias (MS, 2019–2023)",
    labels={"indicador": "Indicador", "valor": "Taxa media", "ano": "Ano"},
    color_discrete_map={
        "Crescimento do PIB":       "#2E75B6",
        "Crescimento Populacional": "#70AD47",
    },
)
fig_taxas.update_traces(marker_size=9, line_width=2.5)
fig_taxas.add_vline(
    x=2020, line_dash="dot", line_color="gray", line_width=1.5,
    annotation_text="2020 (COVID-19)",
    annotation_position="top left",
    annotation_font_size=11,
)
fig_taxas.update_yaxes(tickformat=".1%", gridcolor="#eeeeee")
fig_taxas.update_xaxes(tickmode="linear", dtick=1)
fig_taxas.update_layout(
    plot_bgcolor="white",
    legend=dict(orientation="h", yanchor="bottom", y=-0.3, xanchor="left", x=0),
    yaxis_title="Taxa media anual",
    xaxis_title="Ano",
)
fig_taxas.show()


## 6. Distribuicao das variaveis principais

In [12]:
base_box = base.copy()
base_box["Porte"] = base_box["dummy_pequeno_municipio"].map({0: "Grande", 1: "Pequeno"})

fig = px.box(
    base_box,
    x="ano",
    y="gasto_pessoal_pib",
    color="Porte",
    points="all",
    notched=True,
    color_discrete_map={"Grande": "#2980b9", "Pequeno": "#e74c3c"},
    title="Distribuição do gasto com pessoal / PIB por porte e ano (MS, 2019–2023)",
    labels={
        "gasto_pessoal_pib": "Gasto c/ Pessoal / PIB",
        "ano": "Ano",
        "Porte": "Porte do município",
    },
)
fig.update_traces(jitter=0.3, marker_size=4, marker_opacity=0.5)
fig.update_yaxes(tickformat=".0%", gridcolor="#eeeeee")
fig.update_layout(
    plot_bgcolor="white",
    legend=dict(orientation="h", yanchor="bottom", y=-0.25, xanchor="left", x=0),
    xaxis_title="Ano",
    yaxis_title="Gasto c/ Pessoal / PIB",
)
fig.show()

In [13]:
media_geral = base["crescimento_pib"].mean()

fig = px.histogram(
    base,
    x="crescimento_pib",
    color="ano",
    nbins=20,
    barmode="overlay",
    opacity=0.60,
    title="Distribuição da taxa de crescimento do PIB municipal (MS, 2019–2023)",
    labels={"crescimento_pib": "Taxa de crescimento do PIB", "ano": "Ano"},
    color_discrete_sequence=px.colors.qualitative.Set2,
)

# Linha no zero — anotação à esquerda da linha
fig.add_vline(x=0, line_dash="dash", line_color="black", line_width=2)
fig.add_annotation(
    x=0,
    y=0.97,
    yref="paper",
    text="zero",
    showarrow=False,
    xanchor="right",
    xshift=-6,
    font=dict(color="black", size=11),
    bgcolor="white",
    bordercolor="black",
    borderpad=3,
    borderwidth=1,
)

# Linha na média — anotação à direita da linha
fig.add_vline(x=media_geral, line_dash="dot", line_color="#e74c3c", line_width=2)
fig.add_annotation(
    x=media_geral,
    y=0.97,
    yref="paper",
    text=f"Média = {media_geral:.1%}",
    showarrow=False,
    xanchor="left",
    xshift=6,
    font=dict(color="#e74c3c", size=11),
    bgcolor="white",
    bordercolor="#e74c3c",
    borderpad=3,
    borderwidth=1,
)

fig.update_xaxes(tickformat=".0%")
fig.update_layout(
    plot_bgcolor="white",
    yaxis=dict(title="Frequência", gridcolor="#eeeeee"),
    xaxis_title="Taxa de crescimento do PIB",
    legend_title="Ano",
    bargap=0.05,
    height=420,
)
fig.show()

## 7. Relacao entre variaveis chave e crescimento do PIB

Graficos de dispersao com linha de tendencia OLS (via `numpy.polyfit`) para as principais variaveis explicativas do modelo. Municipios coloridos por porte (pequeno vs grande).

In [14]:
base_sc = base.copy()
base_sc["Porte"] = base_sc["dummy_pequeno_municipio"].map({0: "Grande", 1: "Pequeno"})

fig = px.scatter(
    base_sc,
    x="gasto_pessoal_pib",
    y="crescimento_pib",
    color="Porte",
    size="populacao",
    size_max=28,
    color_discrete_map={"Grande": "#2980b9", "Pequeno": "#e74c3c"},
    hover_name="municipio",
    hover_data={
        "ano": True,
        "populacao": ":,",
        "gasto_pessoal_pib": ":.3f",
        "crescimento_pib": ":.3f",
        "Porte": False,
    },
    facet_col="ano",
    facet_col_wrap=3,
    opacity=0.65,
    title="Gasto com pessoal / PIB × crescimento do PIB — com tendência OLS (MS, 2019–2023)",
    labels={"Porte": "Porte do município"},
)

# Adiciona linha OLS por painel via numpy.polyfit
anos_ordem = sorted(base_sc["ano"].unique())
primeiro_ols = True
for idx, ano in enumerate(anos_ordem):
    df_ano = base_sc[base_sc["ano"] == ano].dropna(
        subset=["gasto_pessoal_pib", "crescimento_pib"]
    )
    x = df_ano["gasto_pessoal_pib"].values
    y = df_ano["crescimento_pib"].values
    if len(x) > 2:
        coef = np.polyfit(x, y, 1)
        x_line = np.linspace(x.min(), x.max(), 60)
        y_line = np.polyval(coef, x_line)
        r = np.corrcoef(x, y)[0, 1]
        row = idx // 3 + 1
        col = idx % 3 + 1
        fig.add_trace(
            go.Scatter(
                x=x_line,
                y=y_line,
                mode="lines",
                line=dict(color="dimgray", width=2, dash="dash"),
                name=f"Tendência OLS (r={r:.2f})" if primeiro_ols else f"r={r:.2f}",
                showlegend=primeiro_ols,
                legendgroup="ols",
            ),
            row=row,
            col=col,
        )
        primeiro_ols = False

fig.update_xaxes(tickformat=".0%", title_text="Gasto c/ Pessoal / PIB")
fig.update_yaxes(tickformat=".0%", title_text="Crescimento do PIB")
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_layout(
    plot_bgcolor="white",
    legend=dict(orientation="h", yanchor="bottom", y=-0.18, xanchor="left", x=0),
)
fig.show()

In [15]:

# Scatter: Y vs X1 (log PIB per capita), X2 (investimento/PIB), X7 (distancia capital)
base_sc2 = base.copy()
base_sc2["Porte"] = base_sc2["dummy_pequeno_municipio"].map({0: "Grande", 1: "Pequeno"})

VARS_SCATTER = [
    ("log_pib_per_capita",  "X1 — log PIB per capita"),
    ("investimento_pib",    "X2 — Investimento / PIB"),
    ("distancia_capital_km","X7 — Distancia da capital (km)"),
]

for xvar, xlabel in VARS_SCATTER:
    df_plot = base_sc2.dropna(subset=[xvar, "crescimento_pib"])
    coef = np.polyfit(df_plot[xvar].values, df_plot["crescimento_pib"].values, 1)
    x_rng = np.linspace(df_plot[xvar].min(), df_plot[xvar].max(), 120)
    y_rng = np.polyval(coef, x_rng)
    r = np.corrcoef(df_plot[xvar].values, df_plot["crescimento_pib"].values)[0, 1]

    fig_sc = px.scatter(
        df_plot,
        x=xvar, y="crescimento_pib",
        color="Porte",
        opacity=0.65,
        color_discrete_map={"Grande": "#2E75B6", "Pequeno": "#ED7D31"},
        hover_name="municipio",
        hover_data={"ano": True, xvar: ":.3f", "crescimento_pib": ":.3f", "Porte": False},
        title=f"Crescimento do PIB vs {xlabel} (r={r:.2f})",
        labels={xvar: xlabel, "crescimento_pib": "Crescimento PIB (Y)"},
    )
    fig_sc.add_scatter(
        x=x_rng, y=y_rng,
        mode="lines", name="Tendencia OLS (todos os anos)",
        line=dict(color="#636363", width=2, dash="dash"),
    )
    fig_sc.update_yaxes(tickformat=".0%", gridcolor="#eeeeee")
    fig_sc.update_layout(
        plot_bgcolor="white",
        legend=dict(orientation="h", yanchor="bottom", y=-0.25, xanchor="left", x=0),
        yaxis_title="Crescimento do PIB (Y)",
        xaxis_title=xlabel,
    )
    fig_sc.show()


## 8. Matriz de correlacao

In [16]:
ABREV = {
    "crescimento_pib":           "Δ PIB",
    "log_pib_per_capita":        "ln(PIB/pop)",
    "investimento_pib":          "Inv/PIB",
    "gasto_pessoal_pib":         "Pessoal/PIB",
    "receita_corrente_pib":      "RC/PIB",
    "crescimento_populacao":     "Δ Pop",
    "area_por_habitante":        "Área/hab",
    "distancia_capital_km":      "Dist. capital",
    "dummy_pequeno_municipio":   "D. pequeno",
    "interacao_pequeno_pessoal": "D×Pessoal",
}

matriz_correlacao = base[variaveis_modelo].corr(numeric_only=True)
correlacao_path = OUTPUTS / "matriz_correlacao.csv"
matriz_correlacao.to_csv(
    correlacao_path, sep=";", decimal=",", encoding="utf-8-sig", index_label="variavel"
)

labels = [ABREV.get(v, v) for v in matriz_correlacao.columns]
z = matriz_correlacao.values

fig = go.Figure(
    go.Heatmap(
        z=z,
        x=labels,
        y=labels,
        colorscale="RdBu",
        zmin=-1,
        zmax=1,
        colorbar=dict(
            title="r de Pearson",
            tickformat=".2f",
            tickvals=[-1, -0.5, 0, 0.5, 1],
        ),
        text=[[f"{v:.2f}" for v in row] for row in z],
        texttemplate="%{text}",
        textfont=dict(size=9),
        hoverongaps=False,
    )
)
fig.update_layout(
    title="Matriz de correlação — variáveis do modelo",
    width=730,
    height=660,
    xaxis=dict(tickangle=-45, side="bottom"),
    plot_bgcolor="white",
)
fig.show()

matriz_correlacao

,crescimento_pib,log_pib_per_capita,investimento_pib,gasto_pessoal_pib,receita_corrente_pib,crescimento_populacao,area_por_habitante,distancia_capital_km,dummy_pequeno_municipio,interacao_pequeno_pessoal
crescimento_pib,1.000000,0.220959,0.000860,-0.187091,-0.149645,0.127442,0.032354,-0.016391,0.057501,-0.022878
log_pib_per_capita,0.220959,1.000000,-0.261721,-0.772417,-0.680658,-0.012050,0.146240,-0.084805,0.225774,-0.031052
investimento_pib,0.000860,-0.261721,1.000000,0.450072,0.569246,-0.016358,0.352174,0.116233,0.319562,0.470910
gasto_pessoal_pib,-0.187091,-0.772417,0.450072,1.000000,0.944385,-0.009142,0.194086,0.139816,0.077814,0.322174
receita_corrente_pib,-0.149645,-0.680658,0.569246,0.944385,1.000000,-0.037660,0.331658,0.130641,0.223218,0.463130
crescimento_populacao,0.127442,-0.012050,-0.016358,-0.009142,-0.037660,1.000000,-0.045792,-0.002661,-0.013589,-0.007108
area_por_habitante,0.032354,0.146240,0.352174,0.194086,0.331658,-0.045792,1.000000,-0.053049,0.490499,0.472231
distancia_capital_km,-0.016391,-0.084805,0.116233,0.139816,0.130641,-0.002661,-0.053049,1.000000,-0.158732,-0.161515
dummy_pequeno_municipio,0.057501,0.225774,0.319562,0.077814,0.223218,-0.013589,0.490499,-0.158732,1.000000,0.886102
interacao_pequeno_pessoal,-0.022878,-0.031052,0.470910,0.322174,0.463130,-0.007108,0.472231,-0.161515,0.886102,1.000000


In [17]:
print(f"Matriz de correlacao salva em: {correlacao_path}")

Matriz de correlacao salva em: C:\Users\marco\Documents\PUC-MINAS\4º Semestre\Eixo 4 - Projeto Mini Ministério da Fazenda\Projeto-Mini-Min-Fazenda\Etapa3\Material_entrega\outputs\matriz_correlacao.csv


## 9. Outliers potenciais

A tabela abaixo usa o criterio do intervalo interquartil para sinalizar observacoes extremas nas variaveis principais. O objetivo aqui e orientar a interpretacao; nao se recomenda excluir observacoes automaticamente sem justificativa economica ou metodologica.

> **Atenção — `crescimento_populacao` em 2023:** Todos os municípios apresentam crescimento populacional zero em 2023. Isso ocorre porque o IBGE não havia publicado as estimativas populacionais municipais para 2023 quando os dados SICONFI foram coletados; o sistema repete a estimativa de 2022. Essa limitação deve ser registrada na seção de limitações do relatório.

In [18]:
# Variaveis avaliadas pelo criterio IQR: apenas as continuas do modelo.
# Excluidas:
#   - dummy_pequeno_municipio    : binaria 0/1, IQR nao se aplica
#   - interacao_pequeno_pessoal  : zero-inflated por construcao (75% zeros), IQR nao se aplica
#   - crescimento_populacao      : limitacao de dados (2023 repete 2022 no IBGE; ver nota na secao 9)
variaveis_outlier = [
    "crescimento_pib",
    "log_pib_per_capita",
    "investimento_pib",
    "gasto_pessoal_pib",
    "receita_corrente_pib",
    "area_por_habitante",
    "distancia_capital_km",
]


def sinalizar_outliers_iqr(df: pd.DataFrame, variavel: str) -> pd.DataFrame:
    q1 = df[variavel].quantile(0.25)
    q3 = df[variavel].quantile(0.75)
    iqr = q3 - q1
    limite_inferior = q1 - 3.0 * iqr
    limite_superior = q3 + 3.0 * iqr
    outliers = df[df[variavel].lt(limite_inferior) | df[variavel].gt(limite_superior)].copy()
    outliers["variavel"] = variavel
    outliers["valor"] = outliers[variavel]
    outliers["limite_inferior"] = limite_inferior
    outliers["limite_superior"] = limite_superior
    return outliers[["ano", "cod_ibge", "municipio", "variavel", "valor", "limite_inferior", "limite_superior"]]


outliers = pd.concat(
    [sinalizar_outliers_iqr(base.dropna(subset=[variavel]), variavel) for variavel in variaveis_outlier],
    ignore_index=True,
)

outliers_path = OUTPUTS / "outliers_potenciais.csv"
outliers.to_csv(outliers_path, index=False, sep=";", decimal=",", encoding="utf-8-sig")
outliers.head(20)

,ano,cod_ibge,municipio,variavel,valor,limite_inferior,limite_superior
0,2022,5004908,Jaraguari,crescimento_pib,1.094741,-0.676724,0.974247
1,2023,5004403,Inocência,crescimento_pib,0.994794,-0.676724,0.974247
2,2019,5000807,Anaurilândia,investimento_pib,0.052267,-0.027518,0.050737
3,2020,5001904,Bataguassu,investimento_pib,0.057399,-0.027518,0.050737
4,2020,5002803,Caracol,investimento_pib,0.072569,-0.027518,0.050737
5,2022,5003108,Corguinho,investimento_pib,0.063450,-0.027518,0.050737
6,2022,5007307,Rio Negro,investimento_pib,0.050786,-0.027518,0.050737
7,2023,5003108,Corguinho,investimento_pib,0.070525,-0.027518,0.050737
8,2023,5003900,Figueirão,investimento_pib,0.052101,-0.027518,0.050737
9,2019,5003900,Figueirão,area_por_habitante,1.599453,-0.794920,1.261288


In [19]:
print(f"Outliers potenciais salvos em: {outliers_path}")
print(f"Quantidade de sinalizacoes: {len(outliers)}")

Outliers potenciais salvos em: C:\Users\marco\Documents\PUC-MINAS\4º Semestre\Eixo 4 - Projeto Mini Ministério da Fazenda\Projeto-Mini-Min-Fazenda\Etapa3\Material_entrega\outputs\outliers_potenciais.csv
Quantidade de sinalizacoes: 15


## 10. Exportacao para Excel formatado

Gera o arquivo nalise_descritiva_resultados.xlsx com quatro abas:

- **Geral**: pivot com as estatisticas da base empilhada (todas as variaveis x todas as estatisticas)
- **Estatisticas**: tabela plana filtrada para os anos 2019-2023 (sem a linha da base empilhada), com nomes completos das variaveis
- **Correlacao**: matriz de correlacao com escala de cores (vermelho-amarelo-verde de -1 a +1)
- **Outliers**: observacoes sinalizadas pelo criterio IQR, valores acima do limite em rosa e abaixo em azul


In [20]:
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.formatting.rule import ColorScaleRule

# ---------- paleta ----------
COR_CABECALHO_BG = "1F3864"
COR_CABECALHO_FG = "FFFFFF"
COR_LINHA_PAR    = "DCE6F1"
COR_LINHA_IMPAR  = "FFFFFF"
COR_CORR_MIN     = "F8696B"
COR_CORR_MED     = "FFEB84"
COR_CORR_MAX     = "63BE7B"
COR_SECAO_BG     = "2E75B6"   # azul medio para cabecalho de secao do pivot

borda_fina = Border(
    left=Side(style="thin", color="BFBFBF"),
    right=Side(style="thin", color="BFBFBF"),
    top=Side(style="thin", color="BFBFBF"),
    bottom=Side(style="thin", color="BFBFBF"),
)

def _cab(size=10, wrap=True):
    return dict(
        font=Font(bold=True, color=COR_CABECALHO_FG, size=size),
        fill=PatternFill("solid", fgColor=COR_CABECALHO_BG),
        alignment=Alignment(horizontal="center", vertical="center", wrap_text=wrap),
        border=borda_fina,
    )

def _ap(cell, **kw):
    for a, v in kw.items():
        setattr(cell, a, v)

def _fw(ws, min_w=8, max_w=42):
    for col in ws.columns:
        w = max((len(str(c.value)) if c.value is not None else 0) for c in col)
        ws.column_dimensions[get_column_letter(col[0].column)].width = min(max(w + 2, min_w), max_w)

# ---------- mapeamentos ----------
NOMES_VARIAVEIS = {
    "crescimento_pib":           "Crescimento do PIB",
    "log_pib_per_capita":        "Log PIB per capita",
    "investimento_pib":          "Investimento / PIB",
    "gasto_pessoal_pib":         "Gasto com Pessoal / PIB",
    "receita_corrente_pib":      "Receita Corrente / PIB",
    "crescimento_populacao":     "Crescimento Populacional",
    "area_por_habitante":        "Área por Habitante",
    "distancia_capital_km":      "Distância da Capital (km)",
    "dummy_pequeno_municipio":   "Dummy Pequeno Município",
    "interacao_pequeno_pessoal": "Interação (Dummy × Pessoal)",
    "populacao":                 "População",
    "pib":                       "PIB (R$)",
    "pib_per_capita":            "PIB per capita (R$)",
    "receita_corrente":          "Receita Corrente (R$)",
    "despesa_pessoal":           "Despesa com Pessoal (R$)",
    "investimento":              "Investimento (R$)",
    "area_km2":                  "Área (km²)",
}

NOMES_ESTATISTICAS = {
    "media":        "Média",
    "erro_padrao":  "Erro Padrão",
    "moda":         "Moda",
    "mediana":      "Mediana",
    "q1":           "1º Quartil",
    "q3":           "3º Quartil",
    "variancia":    "Variância",
    "desvio_padrao":"Desvio Padrão",
    "minimo":       "Mínimo",
    "maximo":       "Máximo",
    "soma":         "Soma",
    "contagem":     "Contagem",
    "ausentes":     "Val. Ausentes",
}

STATS_COLS = list(NOMES_ESTATISTICAS.keys())

FORMATOS_FLAT = {
    "media": "0.0000", "erro_padrao": "0.0000", "moda": "0.0000",
    "mediana": "0.0000", "q1": "0.0000", "q3": "0.0000",
    "variancia": "0.000E+00", "desvio_padrao": "0.0000",
    "minimo": "0.0000", "maximo": "0.0000",
    "soma": "#,##0.00", "contagem": "0", "ausentes": "0",
}

NOMES_PT_FLAT = {
    "grupo": "Grupo", "valor_grupo": "Período", "variavel": "Variável",
    "media": "Média", "erro_padrao": "Erro Padrão", "moda": "Moda",
    "mediana": "Mediana", "q1": "Q1 (25%)", "q3": "Q3 (75%)",
    "variancia": "Variância", "desvio_padrao": "Desvio Padrão",
    "minimo": "Mínimo", "maximo": "Máximo",
    "soma": "Soma", "contagem": "N", "ausentes": "Ausentes",
}

NOMES_OUT = {
    "ano": "Ano", "cod_ibge": "Cod. IBGE", "municipio": "Município",
    "variavel": "Variável", "valor": "Valor",
    "limite_inferior": "Lim. Inferior", "limite_superior": "Lim. Superior",
}

# ============================================================
# Prepara dados
# ============================================================
est_anos  = estatisticas_descritivas[estatisticas_descritivas["grupo"] == "ano"].copy()
est_geral = estatisticas_descritivas[estatisticas_descritivas["grupo"] == "total"].copy()

# Pivot da base_empilhada: linhas=estatisticas, colunas=variaveis
pivot_geral = (
    est_geral
    .set_index("variavel")[STATS_COLS]
    .T
)
pivot_geral.index   = [NOMES_ESTATISTICAS.get(i, i) for i in pivot_geral.index]
pivot_geral.columns = [NOMES_VARIAVEIS.get(c, c) for c in pivot_geral.columns]

wb = openpyxl.Workbook()

# ============================================================
# ABA 1 — Geral (pivot base_empilhada)
# ============================================================
ws_g = wb.active
ws_g.title = "Geral"

# Cabecalho de variaveis
ws_g.cell(1, 1, "Estatística")
_ap(ws_g.cell(1, 1), **_cab())
ws_g.row_dimensions[1].height = 42

for c_idx, col_name in enumerate(pivot_geral.columns, start=2):
    cell = ws_g.cell(1, c_idx, col_name)
    _ap(cell, **_cab(size=9, wrap=True))

# Linhas de estatisticas
for r_idx, (stat_name, row) in enumerate(pivot_geral.iterrows(), start=2):
    cor_bg = COR_LINHA_PAR if r_idx % 2 == 0 else COR_LINHA_IMPAR

    # Celula de rotulo
    lbl = ws_g.cell(r_idx, 1, stat_name)
    _ap(lbl,
        font=Font(bold=True, color=COR_CABECALHO_FG, size=9),
        fill=PatternFill("solid", fgColor=COR_CABECALHO_BG),
        alignment=Alignment(horizontal="left", vertical="center"),
        border=borda_fina,
    )
    ws_g.row_dimensions[r_idx].height = 16

    for c_idx, val in enumerate(row, start=2):
        cell = ws_g.cell(r_idx, c_idx, val if not (hasattr(val, "__float__") and str(val) == "nan") else None)
        cell.fill   = PatternFill("solid", fgColor=cor_bg)
        cell.border = borda_fina
        cell.font   = Font(size=9)
        cell.alignment = Alignment(horizontal="right", vertical="center")
        cell.number_format = "0.0000"

ws_g.freeze_panes = "B2"
ws_g.column_dimensions["A"].width = 18
for i in range(2, len(pivot_geral.columns) + 2):
    ws_g.column_dimensions[get_column_letter(i)].width = 20

# ============================================================
# ABA 2 — Estatisticas (anos apenas, formato plano)
# ============================================================
ws1 = wb.create_sheet("Estatisticas")
cols = list(est_anos.columns)
ws1.append([NOMES_PT_FLAT.get(c, c) for c in cols])
for cell in ws1[1]:
    _ap(cell, **_cab())
ws1.row_dimensions[1].height = 40

for r_idx, row in enumerate(est_anos.itertuples(index=False), start=2):
    cor_bg = COR_LINHA_PAR if r_idx % 2 == 0 else COR_LINHA_IMPAR
    fill = PatternFill("solid", fgColor=cor_bg)
    for c_idx, (cn, val) in enumerate(zip(cols, row), start=1):
        cell = ws1.cell(row=r_idx, column=c_idx, value=val)
        cell.fill   = fill
        cell.border = borda_fina
        cell.font   = Font(size=9)
        cell.alignment = Alignment(
            horizontal="right" if isinstance(val, (int, float)) and not isinstance(val, bool) else "left",
            vertical="center",
        )
        if cn in FORMATOS_FLAT:
            cell.number_format = FORMATOS_FLAT[cn]

ws1.freeze_panes = "D2"
ws1.auto_filter.ref = ws1.dimensions
_fw(ws1)
ws1.column_dimensions["A"].width = 10
ws1.column_dimensions["B"].width = 10
ws1.column_dimensions["C"].width = 30

# ============================================================
# ABA 3 — Correlacao
# ============================================================
ws2 = wb.create_sheet("Correlacao")
variaveis_corr = list(matriz_correlacao.columns)
ws2.append([""] + variaveis_corr)
for cell in ws2[1]:
    if cell.column > 1:
        _ap(cell, **_cab())
        cell.alignment = Alignment(horizontal="center", vertical="bottom", text_rotation=90)
    else:
        cell.fill = PatternFill("solid", fgColor=COR_CABECALHO_BG)
ws2.row_dimensions[1].height = 80

for r_idx, (lbl, row_vals) in enumerate(matriz_correlacao.iterrows(), start=2):
    lc = ws2.cell(r_idx, 1, lbl)
    _ap(lc,
        font=Font(bold=True, color=COR_CABECALHO_FG, size=9),
        fill=PatternFill("solid", fgColor=COR_CABECALHO_BG),
        alignment=Alignment(horizontal="left", vertical="center"),
        border=borda_fina,
    )
    ws2.row_dimensions[r_idx].height = 18
    for c_idx, val in enumerate(row_vals, start=2):
        cell = ws2.cell(r_idx, c_idx, round(float(val), 4))
        cell.number_format = "0.0000"
        cell.border = borda_fina
        cell.alignment = Alignment(horizontal="center", vertical="center")
        cell.font = Font(size=9, bold=(abs(float(val)) == 1.0))

n = len(variaveis_corr)
ws2.conditional_formatting.add(
    "B2:" + get_column_letter(1 + n) + str(1 + n),
    ColorScaleRule(
        start_type="num", start_value=-1, start_color=COR_CORR_MIN,
        mid_type="num",   mid_value=0,   mid_color=COR_CORR_MED,
        end_type="num",   end_value=1,   end_color=COR_CORR_MAX,
    ),
)
ws2.freeze_panes = "B2"
ws2.column_dimensions["A"].width = 28
for i in range(2, n + 2):
    ws2.column_dimensions[get_column_letter(i)].width = 11

# ============================================================
# ABA 4 — Outliers
# ============================================================
ws3 = wb.create_sheet("Outliers")
cols_out = list(outliers.columns)
ws3.append([NOMES_OUT.get(c, c) for c in cols_out])
for cell in ws3[1]:
    _ap(cell, **_cab())
ws3.row_dimensions[1].height = 28

fill_acima  = PatternFill("solid", fgColor="FFD7D7")
fill_abaixo = PatternFill("solid", fgColor="D7E8FF")

for r_idx, row in enumerate(outliers.itertuples(index=False), start=2):
    val   = row.valor
    acima = val > row.limite_superior
    cor_bg = COR_LINHA_PAR if r_idx % 2 == 0 else COR_LINHA_IMPAR
    for c_idx, (cn, cv) in enumerate(zip(cols_out, row), start=1):
        cell = ws3.cell(r_idx, c_idx, cv)
        cell.border = borda_fina
        cell.alignment = Alignment(
            horizontal="right" if isinstance(cv, float) else "left",
            vertical="center",
        )
        if cn == "valor":
            cell.fill = fill_acima if acima else fill_abaixo
            cell.font = Font(size=9, bold=True)
            cell.number_format = "0.0000"
        elif cn in ("limite_inferior", "limite_superior"):
            cell.fill = PatternFill("solid", fgColor=cor_bg)
            cell.font = Font(size=9)
            cell.number_format = "0.0000"
        else:
            cell.fill = PatternFill("solid", fgColor=cor_bg)
            cell.font = Font(size=9)

ws3.freeze_panes = "A2"
ws3.auto_filter.ref = ws3.dimensions
_fw(ws3)
ws3.column_dimensions["C"].width = 26
ws3.column_dimensions["D"].width = 28

# ---------- salvar ----------
xl_path = OUTPUTS / "analise_descritiva_resultados.xlsx"
wb.save(xl_path)
print(f"Excel salvo: {xl_path}")
print(f"  Geral       : {pivot_geral.shape[0]} estatisticas x {pivot_geral.shape[1]} variaveis")
print(f"  Estatisticas: {len(est_anos)} linhas (somente anos)")
print(f"  Correlacao  : {matriz_correlacao.shape[0]}x{matriz_correlacao.shape[1]}")
print(f"  Outliers    : {len(outliers)} linhas")

Excel salvo: C:\Users\marco\Documents\PUC-MINAS\4º Semestre\Eixo 4 - Projeto Mini Ministério da Fazenda\Projeto-Mini-Min-Fazenda\Etapa3\Material_entrega\outputs\analise_descritiva_resultados.xlsx
  Geral       : 13 estatisticas x 17 variaveis
  Estatisticas: 85 linhas (somente anos)
  Correlacao  : 10x10
  Outliers    : 15 linhas


In [21]:

# ============================================================
# ABA 5 — Distribuicao de frequencia do crescimento do PIB
# ============================================================
wb_dist = openpyxl.load_workbook(xl_path)

# Definir faixas de -70% a +120% com intervalos de 10%
bins_vals   = np.arange(-0.70, 1.21, 0.10)
labels_bins = [f"{b:.0%} a {b+0.10:.0%}" for b in bins_vals[:-1]]

base_dist = base[["ano", "crescimento_pib"]].dropna().copy()
base_dist["faixa"] = pd.cut(
    base_dist["crescimento_pib"],
    bins=bins_vals,
    labels=labels_bins,
    right=False,
    include_lowest=True,
)

anos_disp = sorted(base_dist["ano"].unique())
dist_pivot = (
    base_dist.groupby(["faixa", "ano"], observed=True)
    .size()
    .unstack(fill_value=0)
    .reindex(columns=anos_disp, fill_value=0)
)
dist_pivot["Total"]      = dist_pivot.sum(axis=1)
dist_pivot["Perc_Total"] = (dist_pivot["Total"] / dist_pivot["Total"].sum() * 100).round(1)

if "Distribuicao" in wb_dist.sheetnames:
    del wb_dist["Distribuicao"]
ws_d = wb_dist.create_sheet("Distribuicao")

# Cabecalho
cabecalhos = ["Faixa de crescimento"] + [str(a) for a in anos_disp] + ["Total", "% do Total"]
ws_d.append(cabecalhos)
for cell in ws_d[1]:
    _ap(cell, **_cab())
ws_d.row_dimensions[1].height = 28

# Linhas de dados
for r_idx, (faixa, row) in enumerate(dist_pivot.iterrows(), start=2):
    cor_bg = COR_LINHA_PAR if r_idx % 2 == 0 else COR_LINHA_IMPAR
    fill   = PatternFill("solid", fgColor=cor_bg)

    # Celula de faixa
    lbl = ws_d.cell(r_idx, 1, str(faixa))
    _ap(lbl,
        font=Font(size=9),
        fill=fill,
        alignment=Alignment(horizontal="left", vertical="center"),
        border=borda_fina,
    )

    # Contagens por ano
    for c_idx, ano in enumerate(anos_disp, start=2):
        cell = ws_d.cell(r_idx, c_idx, int(row[ano]))
        cell.fill   = fill
        cell.border = borda_fina
        cell.font   = Font(size=9)
        cell.alignment = Alignment(horizontal="center", vertical="center")
        cell.number_format = "0"

    # Total
    col_total = len(anos_disp) + 2
    c_tot = ws_d.cell(r_idx, col_total, int(row["Total"]))
    c_tot.fill   = fill
    c_tot.border = borda_fina
    c_tot.font   = Font(size=9, bold=True)
    c_tot.alignment = Alignment(horizontal="center", vertical="center")
    c_tot.number_format = "0"

    # % do Total
    c_pct = ws_d.cell(r_idx, col_total + 1, row["Perc_Total"] / 100)
    c_pct.fill   = fill
    c_pct.border = borda_fina
    c_pct.font   = Font(size=9)
    c_pct.alignment = Alignment(horizontal="center", vertical="center")
    c_pct.number_format = "0.0%"

    ws_d.row_dimensions[r_idx].height = 16

# Linha de total geral
r_tot = len(dist_pivot) + 2
fill_tot = PatternFill("solid", fgColor=COR_SECAO_BG)
tot_lbl = ws_d.cell(r_tot, 1, "Total Geral")
_ap(tot_lbl,
    font=Font(bold=True, color=COR_CABECALHO_FG, size=9),
    fill=fill_tot,
    alignment=Alignment(horizontal="left", vertical="center"),
    border=borda_fina,
)
for c_idx, ano in enumerate(anos_disp, start=2):
    val = int(dist_pivot[ano].sum())
    c = ws_d.cell(r_tot, c_idx, val)
    _ap(c,
        font=Font(bold=True, color=COR_CABECALHO_FG, size=9),
        fill=fill_tot,
        alignment=Alignment(horizontal="center", vertical="center"),
        border=borda_fina,
    )
    c.number_format = "0"

col_total = len(anos_disp) + 2
c_grand = ws_d.cell(r_tot, col_total, int(dist_pivot["Total"].sum()))
_ap(c_grand,
    font=Font(bold=True, color=COR_CABECALHO_FG, size=9),
    fill=fill_tot,
    alignment=Alignment(horizontal="center", vertical="center"),
    border=borda_fina,
)
c_grand.number_format = "0"

c_pct_tot = ws_d.cell(r_tot, col_total + 1, 1.0)
_ap(c_pct_tot,
    font=Font(bold=True, color=COR_CABECALHO_FG, size=9),
    fill=fill_tot,
    alignment=Alignment(horizontal="center", vertical="center"),
    border=borda_fina,
)
c_pct_tot.number_format = "0.0%"
ws_d.row_dimensions[r_tot].height = 18

# Larguras de coluna
ws_d.column_dimensions["A"].width = 22
for i in range(2, len(cabecalhos) + 1):
    ws_d.column_dimensions[get_column_letter(i)].width = 10
ws_d.freeze_panes = "B2"

wb_dist.save(xl_path)
print(f"Aba 'Distribuicao' adicionada: {xl_path}")
print(f"  Faixas  : {len(dist_pivot)}")
print(f"  Anos    : {anos_disp}")
dist_pivot


Aba 'Distribuicao' adicionada: C:\Users\marco\Documents\PUC-MINAS\4º Semestre\Eixo 4 - Projeto Mini Ministério da Fazenda\Projeto-Mini-Min-Fazenda\Etapa3\Material_entrega\outputs\analise_descritiva_resultados.xlsx
  Faixas  : 16
  Anos    : [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]


ano,2019,2020,2021,2022,2023,Total,Perc_Total
faixa,,,,,,,
-60% a -50%,1,0,0,0,0,1,0.300000
-40% a -30%,0,1,1,0,0,2,0.500000
-30% a -20%,1,0,0,0,1,2,0.500000
-20% a -10%,9,0,0,7,4,20,5.100000
-10% a -0%,22,3,3,13,9,50,12.700000
-0% a 10%,35,9,14,17,15,90,22.800000
10% a 20%,8,19,30,14,15,86,21.800000
20% a 30%,2,20,12,10,20,64,16.200000
30% a 40%,1,16,12,6,6,41,10.400000
